Đây là script chạy phân tích và đánh giá mô hình TAGNet gồm các bước:
- Clone repository từ GitHub
- Load dữ liệu đề bài của cuộc thi và trọng số mô hình
- Chạy phân tích mô hình

**Lưu ý**:
- Script này **không thể** chạy local do ngốn bộ nhớ. Script **không bắt buộc** phải chạy trên GPU, chúng tôi có đính kèm hướng dẫn chi tiết chạy trên [Kaggle](./HOW_TO_RUN_KAGGLE.md).
- Notebook chạy cho mô hình TAGNet của dự án tagflow. Việc thay đổi sang mô hình khác có thể dẫn tới việc notebook không chạy được do sự khác biệt về kiến trúc mô hình.
- Thời gian chạy cụ thể: 20-30 phút
- Với những trường hợp có thời gian chạy quá lớn (>30 phút), chúng tôi khuyến khích nên sử dụng cơ chế chạy offline trên Kaggle. Có thể xem chi tiết kết quả chạy của basline RNN, GRU, LSTM tại [đây](../model/MODEL_RESULTS.md)
- Cell code 10 là cell cần phải chú ý, trước khi chạy bạn phải:
    - Thay đổi đường dẫn đến file data (Do nó thay đổi tùy theo tài khoản Kaggle)
    - Chọn loại mô hình cơ sở muốn chạy (Mặc định là TAGNet và không khuyến khích thay đổi)
    - Chọn trọng số mô hình ở lượt chạy thứ bao nhiêu
    - Toàn bộ ảnh và phân tích trong báo cáo được tạo ra từ script này với MODEL_NAME = "tagnet" và TRIAL = 3

In [ ]:
!git clone https://github.com/CryAndRRich/tagflow.git

In [ ]:
%cd /kaggle/working/tagflow
TAGFLOW_PATH = "/kaggle/working/tagflow"

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install captum --no-deps

In [5]:
import sys

if TAGFLOW_PATH not in sys.path:
    sys.path.append(TAGFLOW_PATH)

In [6]:
import torch
import torch.nn.functional as F

In [7]:
from config import *
from preprocess import *
from model import *
from utils import *
from explainer import *

In [ ]:
# Hàm set_seed định nghĩa tại utils/set_up.py
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [9]:
INPUT_ROOT = "YOUR INPUT ROOT PATH"
WORK_DIR = "/kaggle/working"

# MODEL_NAME có thể nhận giá trị "tagnet", "tarnet", "tacnet", "taanet", "taenet"
# Tuy nhiên do sự khác biệt về kiến trúc mô hình, việc chọn MODEL_NAME != "tagnet" 
# sẽ dẫn đến nguy cơ notebook không chạy được
MODEL_NAME = "tagnet"
# TRIAL có thể nhận giá trị 1, 2, 3, 4
TRIAL = 3 

MODEL_WEIGHTS = f"{TAGFLOW_PATH}/data/weights/{MODEL_NAME}/tagnet_best_model_trial_{TRIAL}.pth"

In [10]:
# Class DataManager được định nghĩa tại preprocess/__init__.py
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED
)

In [11]:
train_loader, val_loader, test_loader = data_manager.get_dataloaders()

In [ ]:
# Hàm update_model_kwargs được định nghĩa tại utils/prepare_model.py
model_kwargs = update_model_kwargs(
    data=data_manager,
    attribute_idx=None,
    model_kwargs=CONFIG_MODEL.MODEL_KWARGS[MODEL_NAME]
)

# Hàm get_model được định nghĩa tại model/models/__init__.py
model = get_model(
    name=MODEL_NAME,
    **model_kwargs  
).to(DEVICE)

model_weights = torch.load(MODEL_WEIGHTS, map_location=DEVICE)
model.load_state_dict(model_weights, strict=False)

In [13]:
# Hàm extract_global_attention được định nghĩa tại explainer/global_attn.py
global_attention_results = extract_global_attention(
    model=model, 
    dataloader=val_loader, 
    id_to_idx=data_manager.id_to_idx, 
    device=DEVICE
)

In [ ]:
for i, (action_id, weight) in enumerate(list(global_attention_results.items())[:5]):
    print(f"Top {i + 1}: Mã hành động {action_id} | Đóng góp trung bình: {weight:.4f}%")

In [ ]:
# Hàm plot_global_attention_area được định nghĩa tại utils/plot_graph.py
plot_global_attention_area(global_attention_results, top_k_mark=50)

In [16]:
# Hàm extract_global_ig được định nghĩa tại explainer/integrate_grad.py
ig_results = extract_global_ig(
    model=model, 
    dataloader=val_loader, 
    task_idx=0, # Có thể nhận các giá trị 0, 1, 2, 3, 4, 5
    id_to_idx=data_manager.id_to_idx, 
    device=DEVICE,
    n_steps=20,         
    max_batches=None    
)

In [ ]:
for i, (action_id, impact) in enumerate(list(ig_results.items())[:10]):
    print(f"Top {i + 1}: Mã hành động {action_id} | Cường độ tác động trung bình: {impact:.4f}")

In [18]:
# Hàm extract_graph_edges được định nghĩa tại explainer/graph_attn.py
edges = extract_graph_edges(
    model=model, 
    dataloader=val_loader, 
    id_to_idx=data_manager.id_to_idx, 
    device=DEVICE,
    layer_idx=0
)

In [ ]:
for i, ((src_action, tgt_action), weight) in enumerate(list(edges.items())[:5]):
    print(f"Luồng {i + 1:02d}: Hành động {src_action} -> Kích hoạt -> Hành động {tgt_action} | Sức mạnh kết nối: {weight:.2f}%")

In [ ]:
# Hàm plot_graph_network được định nghĩa tại utils/plot_graph.py
plot_graph_network(edges, top_k=50, max_occurrences=10)
plot_graph_network(edges, top_k=100, max_occurrences=10)
plot_graph_network(edges, top_k=150, max_occurrences=10)
plot_graph_network(edges, top_k=200, max_occurrences=10)

In [21]:
# Hàm extract_error_attention được định nghĩa tại explainer/error_attn.py
correct_attn, wrong_attn = extract_error_attention(
    model=model, 
    dataloader=val_loader, 
    target_task_idx=5, # Có thể nhận các giá trị 2, 3, 4, 5
    id_to_idx=data_manager.id_to_idx, 
    device=DEVICE
)

In [ ]:
distractors = []
for action_id in wrong_attn:
    if action_id in correct_attn:
        diff = wrong_attn[action_id] - correct_attn[action_id]
        if diff > 0:
            distractors.append((action_id, diff, wrong_attn[action_id], correct_attn[action_id]))

distractors.sort(key=lambda x: x[1], reverse=True)

for i, (action_id, diff, wrong_w, correct_w) in enumerate(distractors[:3]):
    print(f"Top {i + 1}: Mã {action_id}")
    print(f"   - Khi đoán Sai, model tập trung vào nó: {wrong_w:.2f}%")
    print(f"   - Khi đoán Đúng, model chỉ tập trung:   {correct_w:.2f}%")
    print(f"   => Kết luận: Hành vi gây nhiễu, dễ khiến mô hình bị 'ảo tưởng'.")

In [ ]:
# Hàm plot_distractor_analysis được định nghĩa tại utils/plot_graph.py
plot_distractor_analysis(distractors, top_k=5)